## ONLY for use with Google Colab (top cell ONLY)

In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

df_train = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/ToS_summarizer/df_train.csv')
df_val = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/ToS_summarizer/df_val.csv')
df_test = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/ToS_summarizer/df_test.csv')

## Start all the rest of the regular code

In [3]:
# from ToS_preprocessing import df_train, df_val, df_test
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# We need to pull in our cleaned dataframes from the preprocessing notebook
# Load preprocessed splits from CSV
df_train = pd.read_csv('df_train.csv')
df_val = pd.read_csv('df_val.csv')
df_test = pd.read_csv('df_test.csv')

# Confirm GPU availability — if you want to run it in colab for faster computation
# then this will run it with a GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Confirm our splits are still in memory
print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")

Using device: cpu
Train: 5079 | Val: 897 | Test: 1494


In [4]:
# Class definition to reference later for reusability and such
class TOSDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=256):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        # Returns total number of samples — PyTorch needs this
        return len(self.data)

    def __getitem__(self, index):
        # Grab the sentence and label at the given index
        sentence = str(self.data.iloc[index]['sentence'])
        label = int(self.data.iloc[index]['label'])

        # Tokenize the sentence
        # padding/truncation ensures all inputs are the same length
        encoding = self.tokenizer(
            sentence,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(label, dtype=torch.long)
        }

In [5]:
# Load the BERT tokenizer — we'll use 'legal-bert-base-uncased' since our
# text is legalese by defualt, but we can change this to something else if we want
tokenizer = AutoTokenizer.from_pretrained('nlpaueb/legal-bert-base-uncased')

# Instantiate our dataset class for each split
train_dataset = TOSDataset(df_train, tokenizer)
val_dataset = TOSDataset(df_val, tokenizer)
test_dataset = TOSDataset(df_test, tokenizer)

# DataLoaders in pytorch handle batching and shuffling during training
# pin_memory=True speeds up CPU->GPU data transfer when a GPU is available
train_loader = DataLoader(
    train_dataset,
    batch_size=16,          # reduce to 8 if there are memory issues on CPU
    shuffle=True,           # shuffle training data every epoch
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,          # no need to shuffle validation data
    pin_memory=True if torch.cuda.is_available() else False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

Train batches: 318
Val batches: 57
Test batches: 94


In [8]:
# Load Legal-BERT with a classification head on top
# num_labels=3 because we have 3 classes (clearly_fair, potentially_unfair, clearly_unfair)
# We need to explicitly define label mappings so the model knows our 3 classes
# (I got an error the first time because id2label only allows for 2 labels for whatever
# reason unless you map it yourself)
# This handles the id2label mismatch warning
id2label = {0: 'clearly_fair', 1: 'potentially_unfair', 2: 'clearly_unfair'}
label2id = {'clearly_fair': 0, 'potentially_unfair': 1, 'clearly_unfair': 2}

model = AutoModelForSequenceClassification.from_pretrained(
    'nlpaueb/legal-bert-base-uncased',
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True   # bypasses the label mismatch warning
)

model = model.to(device)

# Calculate class weights to handle our slight data imbalance
# The idea is that rarer classes get higher weight so the model doesn't ignore them
class_counts = df_train['label'].value_counts().sort_index().values
class_weights = torch.tensor(
    1.0 / class_counts / (1.0 / class_counts).sum(),
    dtype=torch.float
).to(device)

# Loss function — CrossEntropy with our class weights baked in
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

# AdamW is the standard optimizer for transformer fine-tuning (this is from both this NLP class
# and my Data Science class)
# lr=2e-5 is a safe starting point for BERT from what I can tell
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

# Scheduler gradually reduces learning rate over training
# this helps the model converge more smoothly
total_steps = len(train_loader) * 3  # for each of the 3 epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,  # warm up for the first 10% of steps
    num_training_steps=total_steps
)

print(f"Class weights: {class_weights}")
print(f"Model loaded and ready on: {device}")

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 35403.23it/s]
BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    |

Class weights: tensor([0.2034, 0.3846, 0.4119])
Model loaded and ready on: cpu


In [10]:
# Confirm the model has the correct 3 output labels
print(f"Number of labels: {model.num_labels}")
print(f"id2label: {model.config.id2label}")
print(f"Output layer shape: {model.classifier.weight.shape}")

Number of labels: 3
id2label: {0: 'clearly_fair', 1: 'potentially_unfair', 2: 'clearly_unfair'}
Output layer shape: torch.Size([3, 768])


In [ ]:
def train_epoch(model, dataloader, optimizer, scheduler, loss_fn, device):
    # Set model to training mode for dropout and gradient updates
    model.train()
    total_loss = 0
    correct_predictions = 0

    for batch in dataloader:
        # Move batch tensors to GPU
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        # Zero out gradients from previous batch
        optimizer.zero_grad()

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        # Calculate loss using our weighted loss function
        loss = loss_fn(logits, labels)

        # Track correct predictions for accuracy
        preds = torch.argmax(logits, dim=1)
        correct_predictions += torch.sum(preds == labels)
        total_loss += loss.item()

        # Backward pass — compute gradients
        loss.backward()

        # Clip gradients to prevent exploding gradient problem
        # common practice for transformer fine-tuning (AI suggestion here)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # Update weights and learning rate
        optimizer.step()
        scheduler.step()

    avg_loss = total_loss / len(dataloader)
    accuracy = correct_predictions.double() / len(dataloader.dataset)
    return avg_loss, accuracy


def eval_epoch(model, dataloader, loss_fn, device):
    # Set model to evaluation mode — disables dropout
    model.eval()
    total_loss = 0
    correct_predictions = 0

    # Disable gradient computation for evaluation — saves on memory and speeds things up
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            loss = loss_fn(logits, labels)
            preds = torch.argmax(logits, dim=1)
            correct_predictions += torch.sum(preds == labels)
            total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    accuracy = correct_predictions.double() / len(dataloader.dataset)
    return avg_loss, accuracy